In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [3]:
from pyspark.sql.functions import *

# Capital Gain Loss

## Problem Description

You are given a PySpark DataFrame named **`stocks`** containing stock trading operations.

The DataFrame has the following columns:

| Column | Data Type | Description |
|---|---|---|
| `stock_name` | string | Name of the stock |
| `operation` | string | Type of operation: `Buy` or `Sell` |
| `operation_day` | int | Day on which the operation occurred |
| `price` | int | Price of the stock during the operation |

### Task

Calculate the **capital gain or loss** for each stock.

The capital gain/loss is calculated as:

```text
Capital Gain/Loss = Total Sell Price - Total Buy Price

In [4]:
data = [
    ("Leetcode", "Buy", 1, 1000),
    ("Leetcode", "Sell", 5, 9000),
    ("Leetcode", "Buy", 6, 4000),
    ("Leetcode", "Sell", 8, 5000),

    ("Corona", "Buy", 2, 3000),
    ("Corona", "Sell", 3, 1200),

    ("Amazon", "Buy", 4, 5000),
    ("Amazon", "Sell", 7, 8000),
    ("Amazon", "Buy", 10, 2000),
    ("Amazon", "Sell", 12, 1500),

    ("Tesla", "Buy", 3, 7000),
    ("Tesla", "Sell", 6, 9000),
    ("Tesla", "Buy", 9, 3000),
    ("Tesla", "Sell", 11, 2500),

    ("Google", "Buy", 2, 4000),
    ("Google", "Sell", 5, 6000),

    ("Microsoft", "Buy", 1, 8000),
    ("Microsoft", "Sell", 4, 7500),
    ("Microsoft", "Buy", 8, 2000),
    ("Microsoft", "Sell", 10, 3500),
]

columns = [
    "stock_name",
    "operation",
    "operation_day",
    "price"
]

stocks = spark.createDataFrame(data, columns)

stocks.show()

+----------+---------+-------------+-----+
|stock_name|operation|operation_day|price|
+----------+---------+-------------+-----+
|  Leetcode|      Buy|            1| 1000|
|  Leetcode|     Sell|            5| 9000|
|  Leetcode|      Buy|            6| 4000|
|  Leetcode|     Sell|            8| 5000|
|    Corona|      Buy|            2| 3000|
|    Corona|     Sell|            3| 1200|
|    Amazon|      Buy|            4| 5000|
|    Amazon|     Sell|            7| 8000|
|    Amazon|      Buy|           10| 2000|
|    Amazon|     Sell|           12| 1500|
|     Tesla|      Buy|            3| 7000|
|     Tesla|     Sell|            6| 9000|
|     Tesla|      Buy|            9| 3000|
|     Tesla|     Sell|           11| 2500|
|    Google|      Buy|            2| 4000|
|    Google|     Sell|            5| 6000|
| Microsoft|      Buy|            1| 8000|
| Microsoft|     Sell|            4| 7500|
| Microsoft|      Buy|            8| 2000|
| Microsoft|     Sell|           10| 3500|
+----------

# Using Spark Sql

In [5]:
stocks.createOrReplaceTempView("stocks")

In [22]:
spark.sql(
    """
    with cte as (
        SELECT stock_name,operation, SUM(price) as total
        from stocks 
        GROUP BY stock_name,operation)
    """
    
).show()


AnalysisException: [MISSING_AGGREGATION] The non-aggregating expression "stock_name" is based on columns which are not participating in the GROUP BY clause.
Add the columns or the expression to the GROUP BY, aggregate the expression, or use "any_value(stock_name)" if you do not care which of the values within a group is returned.;
WithCTE
:- CTERelationDef 13, false
:  +- SubqueryAlias cte
:     +- Aggregate [stock_name#0, operation#1], [stock_name#0, operation#1, sum(price#3L) AS total#164L]
:        +- SubqueryAlias stocks
:           +- View (`stocks`, [stock_name#0,operation#1,operation_day#2L,price#3L])
:              +- LogicalRDD [stock_name#0, operation#1, operation_day#2L, price#3L], false
+- Aggregate [operation#1], [stock_name#0, operation#1, total#164L]
   +- SubqueryAlias CTE
      +- CTERelationRef 13, true, [stock_name#0, operation#1, total#164L]
